# PART IV: Clustering

In [ ]:
import sklearn as sk
import pandas as pd

Clustering Techniques: Use the scikit learn clustering library: https://scikit-learn.org/stable/modules/clustering.htmlLinks to an external site. Note that this page contains very useful illustrations and overviews of the clustering methods, worth reading!
K-means clustering: cluster.KMeansLinks to an external site..
Hierarchical clustering: Experiment with "ward", "complete", "average", and "single" linkage: cluster.AgglomerativeClusteringLinks to an external site..
DBSCAN clustering: Run experiments to find good parameter values. cluster.DBSCANLinks to an external site..

## Importing data

In [ ]:
df = pd.read_parquet("advanced_feature_selected_data.parquet")
df.info()

## Data Pre-processing

Scale all continuous attributes to a 0 to 1 scale using MinMaxScaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler

numeric_columns = df.select_dtypes(include="number").columns
df[numeric_columns] = MinMaxScaler().fit_transform(df[numeric_columns])

Take 1% subset

In [ ]:
from sklearn.model_selection import train_test_split

sample_df, _ = train_test_split(
    df,
    train_size=0.01,
    stratify=df["Severity"],
    random_state=42,
)

X = sample_df.drop(columns=["Severity"])
y = sample_df["Severity"]

Drop boolean values

In [ ]:
boolean_columns = X.select_dtypes(include="boolean").columns
X.drop(columns = boolean_columns, inplace = True)

## Clustering

In [ ]:
from sklearn.base import clone
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
import matplotlib.pyplot as plt
import numpy as np
import umap

def make_cluster_plot(model, embedding, ax, title=None):
    cluster_labels = np.asarray(model.labels_)
    unique_labels, color_values = np.unique(cluster_labels, return_inverse=True)

    cmap = plt.get_cmap("turbo", len(unique_labels))
    scatter = ax.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=color_values,
        cmap=cmap,
        s=12,
        alpha=0.75,
    )
    ax.set(
        xlabel="UMAP 1",
        ylabel="UMAP 2",
        title=title or "Clusters in UMAP space",
    )

    colorbar = ax.figure.colorbar(scatter, ax=ax, ticks=np.arange(len(unique_labels)))
    if len(unique_labels) <= 20:
        colorbar.ax.set_yticklabels(
            ["Noise (-1)" if label == -1 else str(label) for label in unique_labels]
        )
    colorbar.set_label("Cluster")
    


def make_cluster_plot_panel(models, X, labels=None):
    models = list(models)
    if labels is None:
        labels = [None] * len(models)

    embedding = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        random_state=42,
    ).fit_transform(X)

    columns = min(2, len(models))
    rows = (len(models) + columns - 1) // columns
    fig, axes = plt.subplots(
        rows, columns, figsize=(8 * columns, 5 * rows), squeeze=False
    )

    for model, label, ax in zip(models, labels, axes.flat):
        make_cluster_plot(model, embedding, ax, label)

    for ax in axes.flat[len(models):]:
        ax.remove()

    fig.suptitle("Clustering assignments visualized with UMAP")
    fig.tight_layout()
    return fig, axes


def compare_clustering_parameter(X, estimator, parameter, values, **fixed_params):
    base_model = estimator(**fixed_params)
    values = list(values)

    models = [
        clone(base_model).set_params(**{parameter: value}).fit(X)
        for value in values
    ]
    labels = [
        f"{type(base_model).__name__}: {parameter}={value}"
        for value in values
    ]
    return make_cluster_plot_panel(models, X, labels), [l for l in labels], [m for m in models]

### K-means Clustering

In [ ]:
k_values = range(1, 30)
sse = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=0, n_init=10).fit(X)
    sse.append(model.inertia_)

plt.figure(figsize=(10, 4))
plt.plot(k_values, sse, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("SSE (inertia)")
plt.title("K-means elbow plot")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

In [ ]:
cluster_plot_fig, labels, models = compare_clustering_parameter(
    X,
    KMeans,
    parameter="n_clusters",
    values=[2, 5, 8, 10],
    random_state = 0
)
plt.show()

In [ ]:
def get_results(labels, models):
    stats = []
    for label, model in zip(labels, models):
        pct_dict = (
            (pd.Series(model.labels_).value_counts(normalize=True) * 100)
            .round(1)
            .to_dict()
        )

        stats.append(
            {
                "Model": label,
                "Iterations": model.n_iter_,
                "% per Cluster": pct_dict,
                "SSE": model.inertia_
            }
        )

    return pd.DataFrame(stats)

results_df = get_results(labels, models)
print(results_df.to_string(index=False))

In [ ]:
def analyze_kmeans_anomalies(kmeans_model, X, percentile=99):
    scores = np.min(kmeans_model.transform(X), axis=1)
    
    threshold = np.percentile(scores, percentile)
    outliers = scores > threshold
    X_values = X.to_numpy()
    
    plt.figure(figsize=(8, 5))
    plt.scatter(X_values[~outliers, 0], X_values[~outliers, 1], c='blue', alpha=0.5, label='Normal')
    plt.scatter(X_values[outliers, 0], X_values[outliers, 1], c='red', marker='x', s=80, label='Outliers')
    plt.scatter(kmeans_model.cluster_centers_[:, 0], kmeans_model.cluster_centers_[:, 1], 
                c='black', marker='*', s=200, label='Centroids')
    plt.title(f"K-Means Anomalies (Top {100-percentile}%)")
    plt.legend()
    plt.show()

    print(f"Outliers Identified: {outliers.sum()} (out of {len(X)} total points)")
    print(f"Distance Threshold f(x): {threshold:.2f}\n")
    
    if outliers.any():
        print("Outlier Coordinates:")
        print(np.round(X_values[outliers], 2))

    return outliers

for label, model in zip(labels, models):
    print(label)
    analyze_kmeans_anomalies(model, X)

In [ ]:
cluster_plot_fig, labels, models= compare_clustering_parameter(
    X,
    KMeans,
    parameter="n_init",
    values=[1, 5, 10, 20],
    random_state = 0
)
plt.show()

In [ ]:
results_df = get_results(labels, models)
print(results_df.to_string(index=False))

In [ ]:
for label, model in zip(labels, models):
    print(label)
    analyze_kmeans_anomalies(model, X)

In [ ]:
cluster_plot_fig, labels, models = compare_clustering_parameter(
    X,
    KMeans,
    parameter="max_iter",
    values=[1, 10],
    random_state = 0
)
plt.show()

In [ ]:
results_df = get_results(labels, models)
print(results_df.to_string(index=False))

## Agglomerative Clustering

In [ ]:
from sklearn.neighbors import kneighbors_graph

#reduce memory use + runtime by forcing evaluation of only 10 nearest neighbors
connectivity = kneighbors_graph(X, n_neighbors=100, include_self=False)

cluster_plot_fig, labels, models= compare_clustering_parameter(
    X,
    AgglomerativeClustering,
    parameter="linkage",
    values=['ward', 'average', 'complete', 'single'],
    connectivity = connectivity
)
plt.show()

## DBSCAN

In [ ]:
from sklearn.neighbors import NearestNeighbors

min_samples_values = [5, 10, 20, 50]
min_samples_values = [m for m in min_samples_values if m < len(X)]

fig, ax = plt.subplots(figsize=(8, 5))
all_k_distances = []

for min_samples in min_samples_values:
    distances, _ = NearestNeighbors(n_neighbors=min_samples).fit(X).kneighbors(X)
    k_distances = np.sort(distances[:, -1])
    all_k_distances.extend(k_distances)
    ax.plot(k_distances, label=f"min_samples={min_samples}")

ax.set(
    xlabel="Points sorted by distance",
    ylabel="Distance to k-th neighbor",
    title="DBSCAN k-distance curves",
)
ax.legend()
ax.grid(True)
plt.show()

In [ ]:

cluster_plot_fig, labels, models = compare_clustering_parameter(
    X,
    DBSCAN,
    parameter="algorithm",
    values=['auto', 'ball_tree', 'kd_tree', 'brute']
)
plt.show()

In [ ]:

cluster_plot_fig, labels, models = compare_clustering_parameter(
    X,
    DBSCAN,
    parameter="eps",
    values=[0.2, 0.3, 0.4]
)
plt.show()

In [ ]:

cluster_plot_fig, labels, models = compare_clustering_parameter(
    X,
    DBSCAN,
    parameter="min_samples",
    values=[10, 50, 200, 1000]
)
plt.show()